In [2]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv

import os
from langchain_huggingface import HuggingFaceEmbeddings
import faiss

load_dotenv()

True

In [3]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "ConversationalAIBot"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["HUGGINGFACE_API_KEY"] = os.getenv("HUGGINGFACE_API_KEY")

Embedding Model

In [4]:
#Vector Embeddings (Embedding Model Output features : 384)

from langchain.embeddings import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2", model_kwargs = {"device" : "cpu"})
embedding_model

C:\Users\bharath.sr.lv\AppData\Local\Temp\ipykernel_22796\3273188620.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2", model_kwargs = {"device" : "cpu"})
c:\Users\bharath.sr.lv\Desktop\ConversationalAI\ConversationalAI3.10\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={'device': 'cpu'}, encode_kwargs={}, multi_process=False, show_progress=False)

LLM Model

In [5]:
model = ChatGroq(temperature=0, model_name="deepseek-r1-distill-llama-70b", streaming=True)

In [6]:
#Check 

print(model.invoke("Hi Google").content)

<think>

</think>

Hello! How can I assist you today? 😊


Loading Vector Database

In [7]:
#Local
from langchain_community.vectorstores import FAISS
from langchain_community.docstore import InMemoryDocstore

local_vector_db = FAISS.load_local("C:/Users/bharath.sr.lv/Desktop/ConversationalAI/Conversational_Ai/Data1/conversation_faiss_index",
                                    embeddings=embedding_model,
                                    allow_dangerous_deserialization=True)

In [8]:
vector_db = local_vector_db.as_retriever(search_kwargs={"k":2})

vector_db.invoke("I am facing internet issues and my screen is buffereing")

[Document(id='00a55f1e-6944-4b10-80fe-c60cccbeeb19', metadata={'intent': 'report_poor_signal_coverage'}, page_content='"Instruction: \n    i got to report poor internet signal covedrage i need help\n    Intent: report_poor_signal_coverage\n    Category: COMPLAINTS\n    Tags: BCLQZ\n    Response: We recognize the importance of reliable signal coverage for effective communication. Please follow the steps below to report any issues you are experiencing with poor signal coverage:\r\n\r\n1. Visit our website at {{WEBSITE_URL}}.\r\n2. Navigate to the {{SUPPORT_SECTION}} section.\r\n3. Select the option labeled {{REPORT_POOR_SIGNAL_COVERAGE_OPTION}}.\r\n4. Fill in the required details, including your location and issue description.\r\n5. Submit the form to notify our team.\r\n\r\nOur team will assess your submission and take appropriate measures to enhance network coverage in your vicinity.'),
 Document(id='3730584b-f5d3-42ce-91de-03dbf2c36c02', metadata={'intent': 'report_poor_signal_coverag

#### Lang Graph Workflow

For Intent classification Using Pydantic output Parser

In [9]:
#Using pydantic

from pydantic import BaseModel, Field
from langchain.output_parsers import PydanticOutputParser

class Intent_Classifier(BaseModel):
    """Intent Classifier"""
    customer_id: str = Field(description="The customer ID in the format CUSTXXXX")
    intent: str = Field(description="Intent of the user query")
    query: str = Field(description="User query")
    Reasoning: str = Field(description='Reasoning behind topic selection')


parser = PydanticOutputParser(pydantic_object=Intent_Classifier)

In [10]:
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"description": "Intent Classifier", "properties": {"customer_id": {"description": "The customer ID in the format CUSTXXXX", "title": "Customer Id", "type": "string"}, "intent": {"description": "Intent of the user query", "title": "Intent", "type": "string"}, "query": {"description": "User query", "title": "Query", "type": "string"}, "Reasoning": {"description": "Reasoning behind topic selection", "title": "Reasoning", "type": "string"}}, "required": ["customer_id", "intent", "query", "Reasoning"]}
```


Creating State Graph using Langflow

In [11]:
import operator
from typing import List
from langchain.prompts import PromptTemplate
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph,MessagesState,START,END
from langgraph.prebuilt import ToolNode

In [12]:
llm = ChatGroq(temperature=0, model_name="deepseek-r1-distill-llama-70b", streaming=True)

In [13]:
#Defining State
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

Supervisor Node

In [35]:
from langchain.prompts import PromptTemplate

def supervisor_node(state: AgentState) -> AgentState:
    print("--------------------------Supervisor---------------------------")
    user_question = state["messages"][-1].content

    template  = """
        You are a telecom assistant that classifies the user's intent into one of the following:
    - Plan: If the user is asking about mobile, broadband, or 5G plans.
    - Complaint: If the user is reporting issues like network outage, slow internet, or billing problems.
    - Other: For greetings, general questions, or unrelated topics.

    Your job is to classify the intent and explain your reasoning.

    User Query: {question}

    {format_instructions}
    """
    prompt = PromptTemplate(
        input_variable = ["question"],
        partial_variables={"format_instructions": parser.get_format_instructions()},
        template = template
    )

    chain = prompt | llm | parser

    response  = chain.invoke({"question":user_question})

    print("Parsed Response: ", response)

    return {
        "messages": [
            AIMessage(content=response.intent),
            HumanMessage(content=response.Reasoning)
        ],
        "customer_id": response.customer_id,
        "user_query": response.query,
        "intent": response.intent
    }


In [36]:
state["messages"]

[HumanMessage(content='My Plan is not sufficient and i need more data what can I do My ID is CUST1001?', additional_kwargs={}, response_metadata={})]

In [37]:
#check
state = {
    "messages": [
        HumanMessage(content="My Plan is not sufficient and i need more data what can I do My ID is CUST1001?")
    ]
}
supervisor_node(state)

# state = {
#     "messages": [
#         HumanMessage(content="I am facing buffering issue with my plan and there is no speed in 5G band and though I pay for 5G")
#     ]
# }
# supervisor_node(state)

--------------------------Supervisor---------------------------
Parsed Response:  customer_id='CUST1001' intent='Plan' query='My Plan is not sufficient and i need more data what can I do My ID is CUST1001?' Reasoning='The user is discussing their current plan and the need for more data, indicating they are looking to adjust their plan.'


{'messages': [AIMessage(content='Plan', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='The user is discussing their current plan and the need for more data, indicating they are looking to adjust their plan.', additional_kwargs={}, response_metadata={})],
 'customer_id': 'CUST1001',
 'user_query': 'My Plan is not sufficient and i need more data what can I do My ID is CUST1001?',
 'intent': 'Plan'}

Creating SQL Agent

In [17]:
from langchain_community.utilities import SQLDatabase
from langchain.agents.agent_toolkits import SQLDatabaseToolkit
from langchain.agents import initialize_agent
import re

In [33]:
def SQL_agent(state:AgentState) -> AgentState:
    print("--------------------------SQL Agent---------------------------")

    customer_id = state.get("customer_id")
    # customer_id = "CUST002"
    db = SQLDatabase.from_uri("sqlite:///C:/Users/bharath.sr.lv/Desktop/ConversationalAI/Conversational_Ai/Data1/telecom_customer.db")
    sql_toolkit = SQLDatabaseToolkit(db=db, llm=llm)

    if not customer_id:
        return {
            "messages": [
                AIMessage(content="Customer ID was not found in the Database."),
            ]
        }
    
    sql_agent = initialize_agent(
    tools=sql_toolkit.get_tools(),
    llm=llm,
    agent="zero-shot-react-description",
    verbose=True
    )

    query = f"""
    Provide the customer details for Customer ID '{customer_id}' from the 'customers' table, 
    and return it as a Python dictionary format.
    """
    response = sql_agent.run(query)

    return {"messages": state["messages"] + [AIMessage(content=response)],
            "customer_data":response}
    # print("SQL Agent Output:\n", response)


RAG Agent

In [38]:
from langchain_core.tools import tool

def RAG_agent(state:AgentState) -> AgentState:
    query = state.get("user_query")
    """
    Combines the user query and SQL customer context to retrieve the most relevant information 
    from the vector DB (e.g. troubleshooting guides, plan info, etc.).

    Parameters:
    - query (str): The user's original question (e.g., "Why is my network slow?")
    - customer_context (str): The context retrieved from SQL agent (e.g., customer location, plan, device)

    Returns:
    - str: Top-k retrieved document contents concatenated.
    """
    retriever = local_vector_db.as_retriever(search_kwargs={"k":2})
    documents = retriever.invoke(query)

    context = "\n\n".join([doc.page_content for doc in documents])
    return {
    "messages": state["messages"] + [AIMessage(content=context)],
    "retrieved_context": context
}  